In [ ]:
!pip install ultralytics

In [ ]:
!unzip -t /content/person-bicycle-car.v1i.yolov8.zip

In [ ]:
!unzip -o -q /content/person-bicycle-car.v1i.yolov8.zip -d /content/dataset

In [ ]:
!ls /content/dataset

In [ ]:
!cat /content/dataset/data.yaml

In [ ]:
from pathlib import Path

for category in ['bicycle', 'car', 'person']:
    files = list(Path('/content/dataset/train/labels').glob(f'{category}*.txt'))

    print(f'\n{category}:')
    for f in files[:3]:
        print(f.name, f.read_text()[:150])

In [ ]:
import yaml

yaml_path = '/content/dataset/data.yaml'

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data.update({
    'path': '/content/dataset',
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 3,
    'names': ['bicycle', 'car', 'person']
})

with open(yaml_path, 'w') as f:
    yaml.safe_dump(data, f, sort_keys=False)

In [ ]:
!cat /content/dataset/data.yaml

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project='/content/runs',
    name='person_bicycle_car'
)

In [ ]:
from google.colab import files

files.download('/content/runs/person_bicycle_car/weights/best.pt')

In [ ]:
from IPython.display import Image, display

display(Image('/content/runs/person_bicycle_car/results.png'))

In [ ]:
!ls /content

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/runs/person_bicycle_car/weights/best.pt')

results = model.track(
    source='/content/Video Project 5.mp4',
    tracker='bytetrack.yaml',
    conf=0.25,
    save=True,
    device=0,
    project='/content/runs',
    name='traffic_tracking'
)

In [ ]:
!ls /content/runs/traffic_tracking/

In [ ]:
print(model.ckpt_path)
print(model.names)

In [ ]:
!ffmpeg -y -i "/content/runs/traffic_tracking/Video Project 5.avi" -c:v libx264 -pix_fmt yuv420p /content/traffic_result.mp4

In [ ]:
from IPython.display import Video

Video('/content/traffic_result.mp4', embed=True, width=800)

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/runs/person_bicycle_car/weights/best.pt')

results = model.track(
    source='/content/Video Project 5.mp4',
    tracker='bytetrack.yaml',
    conf=0.1,
    imgsz=960,
    save=True,
    device=0,
    project='/content/runs',
    name='traffic_tracking_v2'
)

In [ ]:
!ffmpeg -y -i "/content/runs/traffic_tracking_v2/Video Project 5.avi" -c:v libx264 -pix_fmt yuv420p /content/traffic_result_v2.mp4

In [ ]:
from IPython.display import Video

Video('/content/traffic_result_v2.mp4', embed=True, width=800)

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/runs/person_bicycle_car/weights/best.pt')

results = model.predict(
    source='/content/Video Project 5.mp4',
    conf=0.1,
    imgsz=960,
    save=True,
    device=0,
    project='/content/runs',
    name='detection_test'
)

In [ ]:
from collections import Counter

counts = Counter()

for r in results:
    for cls in r.boxes.cls:
        counts[model.names[int(cls)]] += 1

print(counts)

In [ ]:
import glob
import random
import matplotlib.pyplot as plt
from PIL import Image

images = glob.glob('/content/dataset/train/images/bicycle*')
samples = random.sample(images, min(12, len(images)))

plt.figure(figsize=(16, 12))

for i, path in enumerate(samples):
    plt.subplot(3, 4, i+1)
    plt.imshow(Image.open(path))
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
!ls /content
!ls /

In [ ]:
import shutil

shutil.move(
    '/bicycle.v1i.yolov8 (1).zip',
    '/content/bicycle_new.zip'
)

In [ ]:
!unzip -o -q /content/bicycle_new.zip -d /content/bicycle_new

In [ ]:
!ls /content/bicycle_new

In [ ]:
!cat /content/bicycle_new/data.yaml

In [ ]:
from pathlib import Path

ids = set()

for f in Path('/content/bicycle_new').rglob('*.txt'):
    if f.parent.name == 'labels':
        for line in f.read_text().splitlines():
            if line.strip():
                ids.add(line.split()[0])

print(ids)

In [ ]:
import shutil

shutil.copytree(
    '/content/dataset',
    '/content/dataset_v2'
)

In [ ]:
for split in ['train', 'valid', 'test']:
    images = list(
        Path(f'/content/bicycle_new/{split}/images').glob('*')
    )
    print(split, ':', len(images))

In [ ]:
from pathlib import Path

for split in ['train', 'valid', 'test']:
    for folder in ['images', 'labels']:
        path = Path(f'/content/dataset_v2/{split}/{folder}')

        for f in path.glob('bicycle*'):
            f.unlink()

print('기존 자전거 데이터 제거 완료')

In [ ]:
import shutil

for split in ['train', 'valid', 'test']:
    for folder in ['images', 'labels']:
        src = Path(f'/content/bicycle_new/{split}/{folder}')
        dst = Path(f'/content/dataset_v2/{split}/{folder}')

        for f in src.iterdir():
            if f.is_file():
                shutil.copy2(f, dst / ('bicycle_new_' + f.name))

print('새로운 자전거 데이터 추가 완료')

In [ ]:
import yaml

yaml_path = '/content/dataset_v2/data.yaml'

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['path'] = '/content/dataset_v2'
data['names'] = ['bicycle', 'car', 'person']

with open(yaml_path, 'w') as f:
    yaml.safe_dump(data, f, sort_keys=False)

print(data)

In [ ]:
for split in ['train', 'valid', 'test']:
    print(f'\n{split}')

    for category in ['bicycle', 'car', 'person']:
        images = list(
            Path(f'/content/dataset_v2/{split}/images')
            .glob(f'{category}*')
        )
        print(category, ':', len(images))

In [ ]:
from pathlib import Path

ids = set()

for f in Path('/content/dataset_v2').rglob('bicycle_new_*.txt'):
    for line in f.read_text().splitlines():
        if line.strip():
            ids.add(line.split()[0])

print(ids)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/dataset_v2/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project='/content/runs',
    name='person_bicycle_car_v2'
)

In [ ]:
from google.colab import files

files.download('/content/runs/person_bicycle_car_v2/weights/best.pt')

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/runs/person_bicycle_car_v2/weights/best.pt')

results = model.track(
    source='/content/Video Project 5.mp4',
    tracker='bytetrack.yaml',
    conf=0.25,
    imgsz=640,
    save=True,
    device=0,
    project='/content/runs',
    name='traffic_tracking_v3'
)

In [ ]:
!ffmpeg -y -i "/content/runs/traffic_tracking_v3/Video Project 5.avi" -c:v libx264 -pix_fmt yuv420p /content/traffic_result_v3.mp4

In [ ]:
from IPython.display import Video

Video('/content/traffic_result_v3.mp4', embed=True, width=800)

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/runs/person_bicycle_car_v2/weights/best.pt')

results = model.track(
    source='/content/Video Project 5.mp4',
    tracker='bytetrack.yaml',
    conf=0.15,
    imgsz=960,
    save=True,
    device=0,
    project='/content/runs',
    name='traffic_tracking_v4'
)

In [ ]:
!ffmpeg -y -i "/content/runs/traffic_tracking_v4/Video Project 5.avi" -c:v libx264 -pix_fmt yuv420p /content/traffic_result_v4.mp4

In [ ]:
from IPython.display import Video

Video('/content/traffic_result_v4.mp4', embed=True, width=800)

In [ ]:
!ffmpeg -y -sseof -5 -i /content/traffic_result_v3.mp4 -t 5 -c:v libx264 -pix_fmt yuv420p /content/final_tracking.mp4

In [ ]:
from IPython.display import Video

Video('/content/final_tracking.mp4', embed=True, width=800)

In [ ]:
from google.colab import files

files.download('/content/final_tracking.mp4')

In [ ]:
files.download('/content/runs/person_bicycle_car_v2/results.png')

In [ ]:
import pandas as pd
from pathlib import Path

data = {}

for category in ['bicycle', 'car', 'person']:
    data[category] = [
        len(list(Path(f'/content/dataset_v2/{split}/images')
                 .glob(f'{category}*')))
        for split in ['train', 'valid', 'test']
    ]

df = pd.DataFrame(data, index=['Train', 'Validation', 'Test']).T
df['Total'] = df.sum(axis=1)
df.loc['Total'] = df.sum()

display(df)

In [ ]:
import yaml
import pandas as pd

with open('/content/runs/person_bicycle_car_v2/args.yaml') as f:
    args = yaml.safe_load(f)

keys = ['model', 'epochs', 'batch', 'imgsz', 'device']

display(pd.DataFrame({
    'Setting': keys,
    'Value': [args[k] for k in keys]
}))